In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

# Adjust the package version to match spark version if needed
spark = SparkSession.builder \
    .appName("KafkaWordCount") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

print("Spark version:", spark.version)


Spark version: 3.5.0


In [7]:
# Create the kafka_df to read from kafka
lines = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "wordcount")
    .option("startingOffsets", "earliest")
    .load()
)

In [8]:
lines.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [9]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import split
from pyspark.sql.functions import expr

lines = lines.withColumn("value", expr("cast(value as string)"))

words = lines.select(
explode(
    split(lines.value, " ")
).alias("word")
)

# Generate running word count
wordCounts = words.groupBy("word").count()
# Start running the query that prints the running counts to the console
query = wordCounts \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()
query.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = 31eae511-6cc3-468b-a588-1a5d4fa0e14b, runId = d1502b56-362b-4327-a818-bc4fec72e3f0] terminated with exception: Failed to create new KafkaAdminClient